# Qugeister - QNN色推定モデル学習

このノートブックでは、QuAic HNN Composerで設計したQNNモデルを学習します。

## ワークフロー
1. QuAic HNN Composerで`config.json`をエクスポート
2. このノートブックで学習を実行
3. `weights.pth`をダウンロードしてQuAicに提出

## 1. 環境セットアップ

In [ ]:
# 必要なライブラリをインストール
!pip install -q pennylane torch numpy tqdm

In [ ]:
import json
import pickle
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
import pennylane as qml
from tqdm.auto import tqdm
from google.colab import files

# デバイス設定
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')

## 2. 設定ファイルのアップロード

QuAic HNN Composerからエクスポートした`config.json`をアップロードしてください。

In [ ]:
# config.jsonをアップロード
print('config.jsonをアップロードしてください...')
uploaded = files.upload()

# 設定を読み込み
config_filename = list(uploaded.keys())[0]
with open(config_filename, 'r') as f:
    config = json.load(f)

print(f'\n設定を読み込みました: {config.get("model_name", "unnamed")}')
print(f'説明: {config.get("description", "N/A")}')

## 3. 棋譜データのダウンロード

QuAicから学習用の棋譜データをダウンロードします。

In [ ]:
# 棋譜データをダウンロード（QuAicのAPIから取得）
import urllib.request
import os

# データセットURL（QuAicから取得）
TRAJECTORY_URL = 'https://quaic.up.railway.app/api/backend/v1/competition/trajectories/diverse_agents_3000/download'
TRAJECTORY_FILE = 'trajectories.pkl'

if not os.path.exists(TRAJECTORY_FILE):
    print('棋譜データをダウンロード中...')
    try:
        urllib.request.urlretrieve(TRAJECTORY_URL, TRAJECTORY_FILE)
        print('ダウンロード完了!')
    except Exception as e:
        print(f'ダウンロードエラー: {e}')
        print('\n手動でアップロードしてください:')
        uploaded_data = files.upload()
        TRAJECTORY_FILE = list(uploaded_data.keys())[0]
else:
    print('棋譜データは既に存在します')

# データを読み込み
with open(TRAJECTORY_FILE, 'rb') as f:
    trajectory_data = pickle.load(f)

print(f'読み込んだサンプル数: {len(trajectory_data)}')

## 4. QNNモデルの構築

config.jsonの設定に基づいてQNNモデルを構築します。

In [ ]:
def build_qnn_from_config(config):
    """config.jsonからQNNモデルを構築"""
    network = config.get('network', {})
    nodes = network.get('nodes', [])
    edges = network.get('edges', [])

    # ノードタイプを抽出
    quantum_nodes = [n for n in nodes if n.get('type') == 'quantum']
    dense_nodes = [n for n in nodes if n.get('type') == 'dense']

    # 量子回路パラメータを取得
    if quantum_nodes:
        q_node = quantum_nodes[0]
        n_qubits = q_node.get('data', {}).get('n_qubits', 4)
        q_circuit = q_node.get('data', {}).get('qCircuit', {})
    else:
        n_qubits = 4
        q_circuit = {}

    print(f'量子ビット数: {n_qubits}')
    print(f'埋め込み: {q_circuit.get("embeddingTemplateName", "AngleEmbedding")}')
    print(f'Ansatz: {q_circuit.get("ansatzTemplateName", "BasicEntanglerLayers")}')

    return n_qubits, q_circuit

n_qubits, q_circuit = build_qnn_from_config(config)

In [ ]:
class ExplicitColorEstimationQNN(nn.Module):
    """QNN色推定モデル

    入力: 448次元 (7チャネル × 64)
    出力: [8, 2] (8駒 × good/bad確率)
    """

    def __init__(self, n_qubits=4, n_layers=2):
        super().__init__()
        self.n_qubits = n_qubits
        self.n_layers = n_layers

        # 前処理層
        self.pre_net = nn.Sequential(
            nn.Linear(448, 64),
            nn.ReLU(),
            nn.Linear(64, n_qubits)
        )

        # 量子デバイス
        self.dev = qml.device('default.qubit', wires=n_qubits)

        # 量子回路の重み
        self.q_weights = nn.Parameter(
            torch.randn(n_layers, n_qubits, 3) * 0.1
        )

        # 後処理層
        self.post_net = nn.Sequential(
            nn.Linear(n_qubits, 32),
            nn.ReLU(),
            nn.Linear(32, 16)  # 8駒 × 2 (good/bad)
        )

        # 量子ノードを定義
        @qml.qnode(self.dev, interface='torch', diff_method='backprop')
        def circuit(inputs, weights):
            # 入力埋め込み（AngleEmbedding）
            qml.AngleEmbedding(inputs, wires=range(n_qubits))

            # Ansatz（BasicEntanglerLayers）
            for layer in range(n_layers):
                for i in range(n_qubits):
                    qml.Rot(weights[layer, i, 0],
                           weights[layer, i, 1],
                           weights[layer, i, 2],
                           wires=i)
                # エンタングル
                for i in range(n_qubits - 1):
                    qml.CNOT(wires=[i, i + 1])
                if n_qubits > 1:
                    qml.CNOT(wires=[n_qubits - 1, 0])

            # 期待値を返す
            return [qml.expval(qml.PauliZ(i)) for i in range(n_qubits)]

        self.circuit = circuit

    def forward(self, x):
        batch_size = x.shape[0]

        # 前処理
        x = self.pre_net(x)

        # 量子回路を適用
        q_out = []
        for i in range(batch_size):
            result = self.circuit(x[i], self.q_weights)
            q_out.append(torch.stack(result))
        q_out = torch.stack(q_out)

        # 後処理
        out = self.post_net(q_out)

        # [batch, 8, 2]にリシェイプしてsoftmax
        out = out.view(batch_size, 8, 2)
        out = torch.softmax(out, dim=-1)

        return out

# モデルを作成
model = ExplicitColorEstimationQNN(n_qubits=n_qubits).to(device)
print(f'\nモデルを作成しました')
print(f'パラメータ数: {sum(p.numel() for p in model.parameters()):,}')

## 5. データの準備

In [ ]:
def prepare_data(trajectory_data, train_ratio=0.8):
    """棋譜データから学習用データを準備"""
    X_list = []
    y_list = []

    for sample in trajectory_data:
        if 'observation' in sample and 'labels' in sample:
            obs = np.array(sample['observation']).flatten()
            labels = np.array(sample['labels'])

            if obs.shape[0] == 448 and labels.shape[0] == 8:
                X_list.append(obs)
                y_list.append(labels)

    X = np.array(X_list, dtype=np.float32)
    y = np.array(y_list, dtype=np.int64)

    # シャッフル
    indices = np.random.permutation(len(X))
    X, y = X[indices], y[indices]

    # 分割
    split_idx = int(len(X) * train_ratio)
    X_train, X_val = X[:split_idx], X[split_idx:]
    y_train, y_val = y[:split_idx], y[split_idx:]

    print(f'学習データ: {len(X_train)} サンプル')
    print(f'検証データ: {len(X_val)} サンプル')

    return X_train, y_train, X_val, y_val

X_train, y_train, X_val, y_val = prepare_data(trajectory_data)

# DataLoader作成
train_dataset = TensorDataset(
    torch.tensor(X_train),
    torch.tensor(y_train)
)
val_dataset = TensorDataset(
    torch.tensor(X_val),
    torch.tensor(y_val)
)

batch_size = config.get('training', {}).get('batch_size', 32)
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=batch_size)

## 6. 学習

In [ ]:
# 学習設定
training_config = config.get('training', {})
epochs = training_config.get('epochs', 50)
learning_rate = training_config.get('learning_rate', 0.001)

print(f'エポック数: {epochs}')
print(f'学習率: {learning_rate}')
print(f'バッチサイズ: {batch_size}')

# オプティマイザと損失関数
optimizer = optim.Adam(model.parameters(), lr=learning_rate)
criterion = nn.CrossEntropyLoss()

In [ ]:
def train_epoch(model, loader, optimizer, criterion, device):
    model.train()
    total_loss = 0
    correct = 0
    total = 0

    for X_batch, y_batch in loader:
        X_batch = X_batch.to(device)
        y_batch = y_batch.to(device)

        optimizer.zero_grad()
        outputs = model(X_batch)  # [batch, 8, 2]

        # 各駒の損失を計算
        loss = 0
        for i in range(8):
            loss += criterion(outputs[:, i, :], y_batch[:, i])
        loss = loss / 8

        loss.backward()
        optimizer.step()

        total_loss += loss.item()

        # 精度計算
        preds = outputs.argmax(dim=-1)  # [batch, 8]
        correct += (preds == y_batch).sum().item()
        total += y_batch.numel()

    return total_loss / len(loader), correct / total


def validate(model, loader, criterion, device):
    model.eval()
    total_loss = 0
    correct = 0
    total = 0

    with torch.no_grad():
        for X_batch, y_batch in loader:
            X_batch = X_batch.to(device)
            y_batch = y_batch.to(device)

            outputs = model(X_batch)

            loss = 0
            for i in range(8):
                loss += criterion(outputs[:, i, :], y_batch[:, i])
            loss = loss / 8

            total_loss += loss.item()

            preds = outputs.argmax(dim=-1)
            correct += (preds == y_batch).sum().item()
            total += y_batch.numel()

    return total_loss / len(loader), correct / total

In [ ]:
# 学習ループ
best_val_acc = 0
history = {'train_loss': [], 'train_acc': [], 'val_loss': [], 'val_acc': []}

print('学習を開始します...\n')

for epoch in tqdm(range(epochs), desc='Training'):
    train_loss, train_acc = train_epoch(model, train_loader, optimizer, criterion, device)
    val_loss, val_acc = validate(model, val_loader, criterion, device)

    history['train_loss'].append(train_loss)
    history['train_acc'].append(train_acc)
    history['val_loss'].append(val_loss)
    history['val_acc'].append(val_acc)

    if val_acc > best_val_acc:
        best_val_acc = val_acc
        torch.save(model.state_dict(), 'best_model.pth')

    if (epoch + 1) % 10 == 0:
        print(f'Epoch {epoch+1}/{epochs}')
        print(f'  Train Loss: {train_loss:.4f}, Acc: {train_acc:.4f}')
        print(f'  Val Loss: {val_loss:.4f}, Acc: {val_acc:.4f}')

print(f'\n学習完了! ベスト検証精度: {best_val_acc:.4f}')

## 7. 学習曲線の可視化

In [ ]:
import matplotlib.pyplot as plt

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

# 損失
ax1.plot(history['train_loss'], label='Train')
ax1.plot(history['val_loss'], label='Validation')
ax1.set_xlabel('Epoch')
ax1.set_ylabel('Loss')
ax1.set_title('Training Loss')
ax1.legend()
ax1.grid(True)

# 精度
ax2.plot(history['train_acc'], label='Train')
ax2.plot(history['val_acc'], label='Validation')
ax2.set_xlabel('Epoch')
ax2.set_ylabel('Accuracy')
ax2.set_title('Training Accuracy')
ax2.legend()
ax2.grid(True)

plt.tight_layout()
plt.savefig('training_history.png', dpi=150)
plt.show()

## 8. モデルのエクスポート

学習済みモデルをダウンロードして、QuAicに提出できます。

In [ ]:
# ベストモデルを読み込み
model.load_state_dict(torch.load('best_model.pth'))

# QuAic提出用にエクスポート
model_name = config.get('model_name', 'qnn_model')
export_filename = f'{model_name}_weights.pth'

torch.save(model.state_dict(), export_filename)
print(f'モデルを保存しました: {export_filename}')

# ダウンロード
print('\nダウンロードを開始...')
files.download(export_filename)
print('\nダウンロード完了!')
print('\n次のステップ:')
print('1. QuAic (https://quaic.up.railway.app) にアクセス')
print('2. 「コンペティション > モデル」に移動')
print('3. ダウンロードした .pth ファイルをアップロード')